# START

In [1]:
# -*- coding: utf-8 -*-
"""
Kvasir-SEG Thesis Pipeline
DINOv3 + Cosine Similarity + Louvain Pruning + PVT-CASCADE

Colab-ready version of the user's UNet++ thesis implementation.
Main change:
    UNet++  ->  PVT-CASCADE

The pruning logic, train/test split, full-vs-pruned experiments,
BCE/structure-style loss, and Dice/IoU evaluation are kept aligned
with the original thesis implementation as much as possible.
"""

"\nKvasir-SEG Thesis Pipeline\nDINOv3 + Cosine Similarity + Louvain Pruning + PVT-CASCADE\n\nColab-ready version of the user's UNet++ thesis implementation.\nMain change:\n    UNet++  ->  PVT-CASCADE\n\nThe pruning logic, train/test split, full-vs-pruned experiments,\nBCE/structure-style loss, and Dice/IoU evaluation are kept aligned\nwith the original thesis implementation as much as possible.\n"

# 0. COLAB INSTALLATION

In [2]:
!pip install -q kagglehub huggingface_hub timm opencv-python scikit-image networkx python-louvain ml-collections

# Clone official CASCADE repository
import os
import sys
import subprocess
from pathlib import Path

CASCADE_DIR = "/content/CASCADE"

if not os.path.exists(CASCADE_DIR):
    !git clone -q https://github.com/SLDGroup/CASCADE.git /content/CASCADE

# IMPORTANT:
# PVT_CASCADE internally loads:
# ./pretrained_pth/pvt/pvt_v2_b2.pth
# Therefore the working directory must be the CASCADE repository.
os.chdir(CASCADE_DIR)

# Download official PVTv2-B2 ImageNet pretrained weights.
os.makedirs("/content/CASCADE/pretrained_pth/pvt", exist_ok=True)

PVT_WEIGHT = "/content/CASCADE/pretrained_pth/pvt/pvt_v2_b2.pth"

if not os.path.exists(PVT_WEIGHT):
    !wget -q -O /content/CASCADE/pretrained_pth/pvt/pvt_v2_b2.pth \
        https://github.com/whai362/PVT/releases/download/v2/pvt_v2_b2.pth

sys.path.insert(0, CASCADE_DIR)

print("CASCADE directory:", os.getcwd())
print("PVT pretrained weights:", os.path.exists(PVT_WEIGHT))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.3 MB/s eta 0:00:00
CASCADE directory: /content/CASCADE
PVT pretrained weights: True


# 1. IMPORTS

In [3]:
# ============================================================
import os
import time
import random
import numpy as np
import networkx as nx
import tifffile

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import transforms

from community import community_louvain

from transformers import AutoImageProcessor, AutoModel
import kagglehub

from lib.networks import PVT_CASCADE

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/CASCADE/lib/pvtv2.py:387: UserWarning: Overwriting pvt_v2_b0 in registry with lib.pvtv2.pvt_v2_b0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/CASCADE/lib/pvtv2.py:397: UserWarning: Overwriting pvt_v2_b1 in registry with lib.pvtv2.pvt_v2_b1. This is because the name being registered conflicts with an existing name. Please check if this is not expect

# 2. CONFIGURATION

In [4]:
# ============================================================
class Config:
    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------
    TRAIN_SPLIT = 0.80
    RANDOM_SEED = 42

    # --------------------------------------------------------
    # PVT-CASCADE
    # --------------------------------------------------------
    IMAGE_SIZE = 224
    BATCH_SIZE = 16
    NUM_WORKERS = 2
    PIN_MEMORY = True

    NUM_CLASSES = 1

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    EPOCHS = 30              # Change to 50/100/200 for final thesis runs
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4

    # --------------------------------------------------------
    # DINOv3 pruning
    # --------------------------------------------------------
    DINO_MODEL_NAME = "rA9del/dinov3b16"

    # Cosine similarity threshold.
    # NOTE: this is cosine similarity directly, not remapped to [0,1].
    SIMILARITY_THRESHOLD = 1 # 0.92, 0.95, 0.97, 1

    # Fraction selected inside each Louvain community.
    RETENTION_RATIO = 0.12 # 0.84, 0.68, 0.44, 0.12

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------
    THRESHOLD = 0.50

    # --------------------------------------------------------
    # Saving
    # --------------------------------------------------------
    OUTPUT_DIR = "/content/thesis_pvt_cascade_outputs"
    MODEL_DIR = "/content/thesis_pvt_cascade_outputs/checkpoints"

    BEST_FULL_MODEL = "best_pvt_cascade_full.pth"
    BEST_PRUNED_MODEL = "best_pvt_cascade_pruned.pth"

    FEATURE_FILE = "/content/thesis_pvt_cascade_outputs/dinov3_features.npy"
    PRUNED_INDEX_FILE = "/content/thesis_pvt_cascade_outputs/pruned_indices.npy"
    SPLIT_FILE = "/content/thesis_pvt_cascade_outputs/train_test_split.npz"


config = Config()

os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.MODEL_DIR, exist_ok=True)

# 3. REPRODUCIBILITY

In [5]:
# ============================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducible behavior.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(config.RANDOM_SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("================================================")
print("DEVICE")
print("================================================")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE
Device: cuda
GPU: Tesla T4


# 4. DOWNLOAD CVC-CLINICDB

In [6]:
# ============================================================
print("\n================================================")
print("DOWNLOADING CVC-CLINICDB")
print("================================================")

dataset_path = kagglehub.dataset_download("balraj98/cvcclinicdb")

print("Dataset path:")
print(dataset_path)

# ------------------------------------------------------------
# Automatically find CVC-ClinicDB image and mask directories
# ------------------------------------------------------------

IMAGE_DIR = None
MASK_DIR = None

for root, dirs, files in os.walk(dataset_path):

    folder_name = os.path.basename(root).lower()

    # Look for Original image folder
    if folder_name == "original":
        IMAGE_DIR = root

    # Look for Ground Truth mask folder
    elif folder_name in ["ground truth", "ground_truth", "groundtruth"]:
        MASK_DIR = root

if IMAGE_DIR is None:
    raise FileNotFoundError(
        f"Could not find the CVC-ClinicDB image directory inside: {dataset_path}"
    )

if MASK_DIR is None:
    raise FileNotFoundError(
        f"Could not find the CVC-ClinicDB mask directory inside: {dataset_path}"
    )

print("Image directory:", IMAGE_DIR)
print("Mask directory :", MASK_DIR)

# ------------------------------------------------------------
# Verify that files actually exist
# ------------------------------------------------------------

image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))
]

mask_files = [
    f for f in os.listdir(MASK_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))
]

print("Number of images:", len(image_files))
print("Number of masks :", len(mask_files))

if len(image_files) == 0:
    raise FileNotFoundError(f"No image files found in: {IMAGE_DIR}")

if len(mask_files) == 0:
    raise FileNotFoundError(f"No mask files found in: {MASK_DIR}")


DOWNLOADING CVC-CLINICDB
Using Colab cache for faster access to the 'cvcclinicdb' dataset.
Dataset path:
/kaggle/input/cvcclinicdb
Image directory: /kaggle/input/cvcclinicdb/PNG/Original
Mask directory : /kaggle/input/cvcclinicdb/PNG/Ground Truth
Number of images: 612
Number of masks : 612


# 5. FIND IMAGE/MASK PAIRS

In [7]:
def prepare_kvasir_data(image_dir, mask_dir):

    valid_ext = (
        ".tif",
        ".tiff",
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp"
    )

    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith(valid_ext)
    ])

    mask_files = sorted([
        f for f in os.listdir(mask_dir)
        if f.lower().endswith(valid_ext)
    ])

    print(f"Images found: {len(image_files)}")
    print(f"Masks found : {len(mask_files)}")

    if len(image_files) == 0:
        raise RuntimeError(
            f"No images found in: {image_dir}"
        )

    if len(mask_files) == 0:
        raise RuntimeError(
            f"No masks found in: {mask_dir}"
        )

    # --------------------------------------------------------
    # CVC-ClinicDB:
    # The image and mask files may not have identical names.
    #
    # Since your dataset contains the same number of images
    # and masks, temporarily pair them according to sorted
    # order.
    # --------------------------------------------------------

    if len(image_files) != len(mask_files):
        raise RuntimeError(
            f"Number of images ({len(image_files)}) "
            f"does not match number of masks ({len(mask_files)})."
        )

    print(f"Matched image/mask pairs: {len(image_files)}")

    # IMPORTANT:
    # Keep returning only image names because your existing
    # KvasirSegDataset expects image_names.
    #
    # We will handle the mask filename inside the Dataset.

    return image_files


all_images = prepare_kvasir_data(
    IMAGE_DIR,
    MASK_DIR
)

print("Total images:", len(all_images))

Images found: 612
Masks found : 612
Matched image/mask pairs: 612
Total images: 612


# 6. DATASET

In [8]:
class KvasirSegDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir,
        image_names,
        image_size=352
    ):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_names = list(image_names)
        self.image_size = image_size

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):

        image_name = self.image_names[idx]

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        mask_path = os.path.join(
            self.mask_dir,
            image_name
        )

        # --------------------------------------------------------
        # Load image
        # --------------------------------------------------------

        try:
            image = Image.open(image_path).convert("RGB")

        except Exception:
            image = Image.fromarray(
                tifffile.imread(image_path)
            ).convert("RGB")

        # --------------------------------------------------------
        # Load mask
        # --------------------------------------------------------

        try:
            mask = Image.open(mask_path).convert("L")

        except Exception:
            mask = Image.fromarray(
                tifffile.imread(mask_path)
            ).convert("L")

        # PVT-CASCADE is tested here at 352x352.
        image = image.resize(
            (self.image_size, self.image_size),
            Image.Resampling.BILINEAR
        )

        # IMPORTANT:
        # Never use bilinear/bicubic interpolation for a binary mask.
        mask = mask.resize(
            (self.image_size, self.image_size),
            Image.Resampling.NEAREST
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        mask = np.asarray(
            mask,
            dtype=np.uint8
        )

        mask = (mask > 127).astype(np.float32)

        image = torch.from_numpy(
            image.transpose(2, 0, 1)
        ).float()

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0).float()

        return image, mask


base_dataset = KvasirSegDataset(
    IMAGE_DIR,
    MASK_DIR,
    all_images,
    image_size=config.IMAGE_SIZE
)

print("Total images:", len(base_dataset))

Total images: 612


# 7. TRAIN / TEST SPLIT

In [9]:
# ============================================================
# This intentionally follows the original thesis implementation:
# 80% training / 20% testing.
#
# The test set is NEVER passed to the pruning algorithm.

indices = np.arange(len(base_dataset))

rng = np.random.default_rng(config.RANDOM_SEED)
rng.shuffle(indices)

split = int(
    config.TRAIN_SPLIT * len(base_dataset)
)

train_indices = indices[:split].tolist()
test_indices = indices[split:].tolist()

print("\n================================================")
print("TRAIN / TEST SPLIT")
print("================================================")
print("Training images:", len(train_indices))
print("Testing images :", len(test_indices))

print(
    "Overlap:",
    len(set(train_indices) & set(test_indices))
)

# Save split so future experiments can reuse exactly the same split.
np.savez(
    config.SPLIT_FILE,
    train_indices=np.asarray(train_indices),
    test_indices=np.asarray(test_indices)
)


TRAIN / TEST SPLIT
Training images: 489
Testing images : 123
Overlap: 0


# 8. DINOv3 GRAPH PRUNER

In [10]:
class AccuracyOptimizedGraphPruner:

    def __init__(
        self,
        dataset,
        train_indices,
        device
    ):

        self.dataset = dataset
        self.train_indices = train_indices
        self.device = device

        # ----------------------------------------------------
        # Initialize community information
        # ----------------------------------------------------

        self.communities = None
        self.community_statistics = None

        # ----------------------------------------------------
        # Initialize graph information
        # ----------------------------------------------------

        self.graph_statistics = None

        # ----------------------------------------------------
        # Read pruning parameters from config
        # ----------------------------------------------------

        self.tau = config.SIMILARITY_THRESHOLD
        self.p = config.RETENTION_RATIO

        print("\n================================================")
        print("LOADING DINOv3")
        print("================================================")

        print(
            f"Similarity threshold (tau): {self.tau}"
        )

        print(
            f"Community retention ratio (p): {self.p}"
        )

        # ----------------------------------------------------
        # Validate pruning parameters
        # ----------------------------------------------------

        if not (0.0 < self.tau <= 1.0):

            raise ValueError(
                f"TAU must be in (0, 1], got {self.tau}"
            )

        if not (0.0 < self.p <= 1.0):

            raise ValueError(
                f"P must be in (0, 1], got {self.p}"
            )

        # ----------------------------------------------------
        # Load DINOv3
        # ----------------------------------------------------

        self.processor = AutoImageProcessor.from_pretrained(
            config.DINO_MODEL_NAME
        )

        self.model = AutoModel.from_pretrained(
            config.DINO_MODEL_NAME
        ).to(self.device)

        self.model.eval()

        # Freeze DINOv3
        for param in self.model.parameters():
            param.requires_grad = False

        print("DINOv3 loaded and frozen.")

    # ========================================================
    # DINOv3 EMBEDDING EXTRACTION
    # ========================================================

    @torch.no_grad()
    def extract_embeddings(self):

        embeddings = []

        print("\n================================================")
        print("EXTRACTING DINOv3 CLS EMBEDDINGS")
        print("================================================")

        for count, idx in enumerate(self.train_indices):

            # ------------------------------------------------
            # Load image
            # ------------------------------------------------

            image, _ = self.dataset[idx]

            # Dataset image should be:
            #
            # [3, H, W]
            #
            # with values in [0, 1]

            image_pil = transforms.ToPILImage()(image)

            # ------------------------------------------------
            # DINOv3 preprocessing
            # ------------------------------------------------

            inputs = self.processor(
                images=image_pil,
                return_tensors="pt"
            )

            inputs = {
                key: value.to(self.device)
                for key, value in inputs.items()
            }

            # ------------------------------------------------
            # Forward pass
            # ------------------------------------------------

            outputs = self.model(**inputs)

            # ------------------------------------------------
            # CLS token representation
            # ------------------------------------------------

            cls_embedding = (
                outputs.last_hidden_state[:, 0, :]
            )

            # ------------------------------------------------
            # L2 normalization
            #
            # After normalization:
            #
            # dot product == cosine similarity
            # ------------------------------------------------

            cls_embedding = F.normalize(
                cls_embedding,
                p=2,
                dim=1
            )

            # ------------------------------------------------
            # Store embedding
            # ------------------------------------------------

            embeddings.append(
                cls_embedding
                .squeeze(0)
                .cpu()
                .numpy()
            )

            # ------------------------------------------------
            # Progress
            # ------------------------------------------------

            if (count + 1) % 100 == 0:

                print(
                    f"Processed {count + 1}/"
                    f"{len(self.train_indices)}"
                )

        # ----------------------------------------------------
        # Convert to NumPy array
        # ----------------------------------------------------

        embeddings = np.asarray(
            embeddings,
            dtype=np.float32
        )

        print("\nEmbedding matrix shape:")
        print(embeddings.shape)

        return embeddings

    # ========================================================
    # GRAPH CONSTRUCTION
    # ========================================================

    def build_similarity_graph(
        self,
        embeddings
    ):

        print("\n================================================")
        print("BUILDING COSINE SIMILARITY GRAPH")
        print("================================================")

        # ----------------------------------------------------
        # Cosine similarity
        #
        # Because embeddings are L2 normalized:
        #
        # cosine(x_i, x_j)
        # =
        # x_i dot x_j
        # ----------------------------------------------------

        cosine_sim = np.matmul(
            embeddings,
            embeddings.T
        )

        # Numerical safety
        cosine_sim = np.clip(
            cosine_sim,
            -1.0,
            1.0
        )

        print(
            "Similarity range:",
            float(cosine_sim.min()),
            "to",
            float(cosine_sim.max())
        )

        # ----------------------------------------------------
        # Build threshold graph
        # ----------------------------------------------------

        binary_edges = (
            cosine_sim >= self.tau
        ).astype(np.int8)

        # ----------------------------------------------------
        # Remove self-loops
        # ----------------------------------------------------

        np.fill_diagonal(
            binary_edges,
            0
        )

        # ----------------------------------------------------
        # Convert adjacency matrix to NetworkX graph
        # ----------------------------------------------------

        G = nx.from_numpy_array(
            binary_edges
        )

        # ----------------------------------------------------
        # Graph statistics
        # ----------------------------------------------------

        num_nodes = G.number_of_nodes()

        num_edges = G.number_of_edges()

        # Maximum possible number of undirected edges

        max_edges = (
            num_nodes * (num_nodes - 1)
        ) / 2

        graph_density = (
            num_edges / max_edges
            if max_edges > 0
            else 0.0
        )

        degrees = [
            degree
            for _, degree in G.degree()
        ]

        average_degree = (
            np.mean(degrees)
            if len(degrees) > 0
            else 0.0
        )

        maximum_degree = (
            np.max(degrees)
            if len(degrees) > 0
            else 0
        )

        isolated_nodes = sum(
            degree == 0
            for degree in degrees
        )

        # ----------------------------------------------------
        # Save graph statistics
        # ----------------------------------------------------

        self.graph_statistics = {

            "nodes": num_nodes,

            "edges": num_edges,

            "density": graph_density,

            "average_degree": average_degree,

            "maximum_degree": maximum_degree,

            "isolated_nodes": isolated_nodes,

            "similarity_threshold": self.tau
        }

        # ----------------------------------------------------
        # Print graph statistics
        # ----------------------------------------------------

        print("\nGraph statistics:")

        print(
            "Nodes              :",
            num_nodes
        )

        print(
            "Edges              :",
            num_edges
        )

        print(
            f"Graph density      : "
            f"{graph_density:.6f}"
        )

        print(
            f"Average degree     : "
            f"{average_degree:.4f}"
        )

        print(
            f"Maximum degree     : "
            f"{maximum_degree}"
        )

        print(
            f"Isolated nodes     : "
            f"{isolated_nodes}"
        )

        return G

    # ========================================================
    # LOUVAIN COMMUNITY DETECTION
    # ========================================================

    def detect_communities(
        self,
        G
    ):

        print("\n================================================")
        print("RUNNING LOUVAIN COMMUNITY DETECTION")
        print("================================================")

        # ----------------------------------------------------
        # Louvain community detection
        # ----------------------------------------------------

        partition = community_louvain.best_partition(
            G,
            random_state=config.RANDOM_SEED
        )

        # ----------------------------------------------------
        # Convert partition into dictionary
        #
        # {
        #     community_id: [node1, node2, ...]
        # }
        # ----------------------------------------------------

        communities = {}

        for node, community_id in partition.items():

            communities.setdefault(
                community_id,
                []
            ).append(node)

        # ----------------------------------------------------
        # Sort nodes inside every community
        # ----------------------------------------------------

        for community_id in communities:

            communities[community_id] = sorted(
                communities[community_id]
            )

        # ----------------------------------------------------
        # Community sizes
        # ----------------------------------------------------

        sizes = [
            len(nodes)
            for nodes in communities.values()
        ]

        # ----------------------------------------------------
        # Community statistics
        # ----------------------------------------------------

        number_of_communities = len(
            communities
        )

        minimum_size = min(
            sizes
        )

        maximum_size = max(
            sizes
        )

        average_size = np.mean(
            sizes
        )

        singleton_count = sum(
            size == 1
            for size in sizes
        )

        # ----------------------------------------------------
        # Store communities
        # ----------------------------------------------------

        self.communities = communities

        # ----------------------------------------------------
        # Print summary
        # ----------------------------------------------------

        print(
            "Number of communities:",
            number_of_communities
        )

        print(
            "Minimum community size :",
            minimum_size
        )

        print(
            "Maximum community size :",
            maximum_size
        )

        print(
            f"Average community size : "
            f"{average_size:.2f}"
        )

        print(
            "Singleton communities  :",
            singleton_count
        )

        # ----------------------------------------------------
        # Print EVERY community size
        # ----------------------------------------------------

        print("\nCommunity sizes:")
        print("-" * 50)

        for community_id in sorted(
            communities.keys()
        ):

            size = len(
                communities[community_id]
            )

            print(
                f"Community {community_id:>3} "
                f": {size:>4} samples"
            )

        print("-" * 50)

        # ----------------------------------------------------
        # Verify total
        # ----------------------------------------------------

        total_community_samples = sum(
            sizes
        )

        print(
            "Total community samples:",
            total_community_samples
        )

        # ----------------------------------------------------
        # Create community statistics
        #
        # Selection values will be filled later.
        # ----------------------------------------------------

        self.community_statistics = []

        for community_id in sorted(
            communities.keys()
        ):

            size = len(
                communities[community_id]
            )

            self.community_statistics.append({

                "community_id":
                    community_id,

                "size":
                    size,

                "selected":
                    0,

                "retention":
                    0.0
            })

        return communities

    # ========================================================
    # REPRESENTATIVE SELECTION
    # ========================================================

    def select_representatives(
        self,
        G,
        communities
    ):

        print("\n================================================")
        print("SELECTING COMMUNITY REPRESENTATIVES")
        print("================================================")

        selected_local_nodes = []

        community_statistics = []

        # ----------------------------------------------------
        # Process communities in sorted order
        # ----------------------------------------------------

        for community_id in sorted(
            communities.keys()
        ):

            nodes = communities[
                community_id
            ]

            community_size = len(
                nodes
            )

            # ------------------------------------------------
            # Singleton community
            # ------------------------------------------------

            if community_size == 1:

                selected_local_nodes.append(
                    nodes[0]
                )

                community_statistics.append({

                    "community_id":
                        community_id,

                    "size":
                        1,

                    "selected":
                        1,

                    "retention":
                        1.0
                })

                continue

            # ------------------------------------------------
            # Calculate degree ONLY inside
            # this community
            # ------------------------------------------------

            subgraph = G.subgraph(
                nodes
            )

            degrees = dict(
                subgraph.degree()
            )

            # ------------------------------------------------
            # Number of samples to retain
            # ------------------------------------------------

            budget = max(
                1,
                int(
                    np.ceil(
                        self.p *
                        community_size
                    )
                )
            )

            # ------------------------------------------------
            # Sort nodes:
            #
            # 1. Highest intra-community degree
            # 2. Lowest node index
            #
            # This makes selection deterministic.
            # ------------------------------------------------

            sorted_nodes = sorted(

                nodes,

                key=lambda node: (
                    -degrees[node],
                    node
                )
            )

            # ------------------------------------------------
            # Select representatives
            # ------------------------------------------------

            selected_nodes = (
                sorted_nodes[:budget]
            )

            selected_local_nodes.extend(
                selected_nodes
            )

            # ------------------------------------------------
            # Store statistics
            # ------------------------------------------------

            community_statistics.append({

                "community_id":
                    community_id,

                "size":
                    community_size,

                "selected":
                    budget,

                "retention":
                    budget /
                    community_size
            })

        # ----------------------------------------------------
        # Sort selected local nodes
        # ----------------------------------------------------

        selected_local_nodes = sorted(
            selected_local_nodes
        )

        # ----------------------------------------------------
        # Save statistics
        # ----------------------------------------------------

        self.community_statistics = (
            community_statistics
        )

        # ----------------------------------------------------
        # Print detailed selection report
        # ----------------------------------------------------

        print("\nCommunity selection details:")
        print("-" * 70)

        print(
            f"{'Community':>12} "
            f"{'Size':>10} "
            f"{'Selected':>12} "
            f"{'Retention':>14}"
        )

        print("-" * 70)

        for item in community_statistics:

            print(
                f"{item['community_id']:>12} "
                f"{item['size']:>10} "
                f"{item['selected']:>12} "
                f"{item['retention'] * 100:>13.2f}%"
            )

        print("-" * 70)

        print(
            "Selected local nodes:",
            len(selected_local_nodes)
        )

        return (
            selected_local_nodes,
            community_statistics
        )

    # ========================================================
    # COMPLETE PRUNING PIPELINE
    # ========================================================

    def prune(self):

        # ----------------------------------------------------
        # 1. Extract DINOv3 embeddings
        # ----------------------------------------------------

        embeddings = (
            self.extract_embeddings()
        )

        # ----------------------------------------------------
        # Save embeddings for thesis analysis
        # ----------------------------------------------------

        np.save(
            config.FEATURE_FILE,
            embeddings
        )

        # ----------------------------------------------------
        # 2. Build similarity graph
        # ----------------------------------------------------

        G = self.build_similarity_graph(
            embeddings
        )

        # ----------------------------------------------------
        # 3. Louvain community detection
        # ----------------------------------------------------

        communities = (
            self.detect_communities(G)
        )

        # ----------------------------------------------------
        # 4. Select representatives
        # ----------------------------------------------------

        (
            selected_local_nodes,
            community_statistics
        ) = self.select_representatives(

            G,

            communities
        )

        # ----------------------------------------------------
        # 5. Map local graph nodes to dataset indices
        # ----------------------------------------------------

        pruned_global_indices = [

            self.train_indices[i]

            for i in selected_local_nodes
        ]

        # ----------------------------------------------------
        # 6. Save selected indices
        # ----------------------------------------------------

        np.save(

            config.PRUNED_INDEX_FILE,

            np.asarray(
                pruned_global_indices,
                dtype=np.int64
            )
        )

        # ----------------------------------------------------
        # 7. Calculate actual global retention
        # ----------------------------------------------------

        original_size = len(
            self.train_indices
        )

        selected_size = len(
            pruned_global_indices
        )

        retention = (
            selected_size /
            original_size
        )

        pruning_rate = (
            1.0 -
            retention
        )

        # ----------------------------------------------------
        # 8. Calculate community totals
        # ----------------------------------------------------

        total_community_samples = sum(
            item["size"]
            for item in community_statistics
        )

        total_community_selected = sum(
            item["selected"]
            for item in community_statistics
        )

        # ----------------------------------------------------
        # 9. Final pruning report
        # ----------------------------------------------------

        print("\n================================================")
        print("PRUNING RESULT")
        print("================================================")

        print(
            "Original training samples:",
            original_size
        )

        print(
            "Graph samples:",
            total_community_samples
        )

        print(
            "Number of communities:",
            len(community_statistics)
        )

        print(
            "Selected training samples:",
            selected_size
        )

        print(
            "Selected community representatives:",
            total_community_selected
        )

        print(
            f"Target community retention (p): "
            f"{self.p:.4f}"
        )

        print(
            f"Actual global retention: "
            f"{retention * 100:.2f}%"
        )

        print(
            f"Actual global pruning rate: "
            f"{pruning_rate * 100:.2f}%"
        )

        print(
            f"Similarity threshold (tau): "
            f"{self.tau:.4f}"
        )

        # ----------------------------------------------------
        # Community summary
        # ----------------------------------------------------

        print("\n================================================")
        print("COMMUNITY SUMMARY")
        print("================================================")

        print(
            f"Number of communities   : "
            f"{len(community_statistics)}"
        )

        print(
            f"Minimum community size  : "
            f"{min(item['size'] for item in community_statistics)}"
        )

        print(
            f"Maximum community size  : "
            f"{max(item['size'] for item in community_statistics)}"
        )

        print(
            f"Average community size  : "
            f"{np.mean([item['size'] for item in community_statistics]):.2f}"
        )

        print(
            f"Singleton communities   : "
            f"{sum(item['size'] == 1 for item in community_statistics)}"
        )

        print("\nCommunity details:")
        print("-" * 70)

        for item in community_statistics:

            print(
                f"Community {item['community_id']:>3} "
                f"| Size: {item['size']:>4} "
                f"| Selected: {item['selected']:>3} "
                f"| Retention: "
                f"{item['retention'] * 100:>6.2f}%"
            )

        print("-" * 70)

        # ----------------------------------------------------
        # Saved files
        # ----------------------------------------------------

        print(
            "\nSaved embeddings to:",
            config.FEATURE_FILE
        )

        print(
            "Saved pruned indices to:",
            config.PRUNED_INDEX_FILE
        )

        # ----------------------------------------------------
        # Store final information in object
        # ----------------------------------------------------

        self.communities = communities

        self.community_statistics = (
            community_statistics
        )

        self.pruned_global_indices = (
            pruned_global_indices
        )

        self.actual_retention = (
            retention
        )

        self.pruning_rate = (
            pruning_rate
        )

        return pruned_global_indices


# 9. RUN PRUNING

In [11]:
# ============================================================
print("\n================================================")
print("DINOv3 + LOUVAIN DATASET PRUNING")
print("================================================")

pruner = AccuracyOptimizedGraphPruner(
    dataset=base_dataset,
    train_indices=train_indices,
    device=device
)

pruned_train_indices = pruner.prune(
    # tau=config.SIMILARITY_THRESHOLD,
    # p=config.RETENTION_RATIO
)


DINOv3 + LOUVAIN DATASET PRUNING

LOADING DINOv3
Similarity threshold (tau): 1
Community retention ratio (p): 0.12


preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/753 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  343MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

DINOv3 loaded and frozen.

EXTRACTING DINOv3 CLS EMBEDDINGS
Processed 100/489
Processed 200/489
Processed 300/489
Processed 400/489

Embedding matrix shape:
(489, 768)

BUILDING COSINE SIMILARITY GRAPH
Similarity range: 0.3912596106529236 to 1.0

Graph statistics:
Nodes              : 489
Edges              : 0
Graph density      : 0.000000
Average degree     : 0.0000
Maximum degree     : 0
Isolated nodes     : 489

RUNNING LOUVAIN COMMUNITY DETECTION
Number of communities: 489
Minimum community size : 1
Maximum community size : 1
Average community size : 1.00
Singleton communities  : 489

Community sizes:
--------------------------------------------------
Community   0 :    1 samples
Community   1 :    1 samples
Community   2 :    1 samples
Community   3 :    1 samples
Community   4 :    1 samples
Community   5 :    1 samples
Community   6 :    1 samples
Community   7 :    1 samples
Community   8 :    1 samples
Community   9 :    1 samples
Community  10 :    1 samples
Community  11 : 

In [12]:
community_statistics = pruner.community_statistics

community_sizes = [
    item["size"]
    for item in community_statistics
]

print(
    "Number of communities:",
    len(community_statistics)
)

for item in community_statistics:

    print(
        f"Community {item['community_id']}: "
        f"{item['size']} samples, "
        f"{item['selected']} selected"
    )


Number of communities: 489
Community 0: 1 samples, 1 selected
Community 1: 1 samples, 1 selected
Community 2: 1 samples, 1 selected
Community 3: 1 samples, 1 selected
Community 4: 1 samples, 1 selected
Community 5: 1 samples, 1 selected
Community 6: 1 samples, 1 selected
Community 7: 1 samples, 1 selected
Community 8: 1 samples, 1 selected
Community 9: 1 samples, 1 selected
Community 10: 1 samples, 1 selected
Community 11: 1 samples, 1 selected
Community 12: 1 samples, 1 selected
Community 13: 1 samples, 1 selected
Community 14: 1 samples, 1 selected
Community 15: 1 samples, 1 selected
Community 16: 1 samples, 1 selected
Community 17: 1 samples, 1 selected
Community 18: 1 samples, 1 selected
Community 19: 1 samples, 1 selected
Community 20: 1 samples, 1 selected
Community 21: 1 samples, 1 selected
Community 22: 1 samples, 1 selected
Community 23: 1 samples, 1 selected
Community 24: 1 samples, 1 selected
Community 25: 1 samples, 1 selected
Community 26: 1 samples, 1 selected
Community 2

In [13]:
# ============================================================
# 7A. LOUVAIN COMMUNITY INFORMATION
# ============================================================

print("\nLOUVAIN COMMUNITY INFORMATION")
print("-" * 70)

print(
    f"Similarity threshold    : "
    f"{config.SIMILARITY_THRESHOLD:.4f}"
)

print(
    f"Number of communities   : "
    f"{len(community_sizes)}"
)

print("\nCommunity sizes:")

for i, size in enumerate(community_sizes, start=1):
    print(
        f"Community {i:2d}            : "
        f"{size} samples"
    )

print(
    f"\nMinimum community size  : "
    f"{min(community_sizes)}"
)

print(
    f"Maximum community size  : "
    f"{max(community_sizes)}"
)

print(
    f"Average community size  : "
    f"{sum(community_sizes) / len(community_sizes):.2f}"
)

print(
    f"Singleton communities   : "
    f"{sum(1 for s in community_sizes if s == 1)}"
)



LOUVAIN COMMUNITY INFORMATION
----------------------------------------------------------------------
Similarity threshold    : 1.0000
Number of communities   : 489

Community sizes:
Community  1            : 1 samples
Community  2            : 1 samples
Community  3            : 1 samples
Community  4            : 1 samples
Community  5            : 1 samples
Community  6            : 1 samples
Community  7            : 1 samples
Community  8            : 1 samples
Community  9            : 1 samples
Community 10            : 1 samples
Community 11            : 1 samples
Community 12            : 1 samples
Community 13            : 1 samples
Community 14            : 1 samples
Community 15            : 1 samples
Community 16            : 1 samples
Community 17            : 1 samples
Community 18            : 1 samples
Community 19            : 1 samples
Community 20            : 1 samples
Community 21            : 1 samples
Community 22            : 1 samples
Community 23            :

# 10. DATALOADERS

In [14]:
# ============================================================
full_train_loader = DataLoader(
    Subset(
        base_dataset,
        train_indices
    ),
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

pruned_train_loader = DataLoader(
    Subset(
        base_dataset,
        pruned_train_indices
    ),
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

test_loader = DataLoader(
    Subset(
        base_dataset,
        test_indices
    ),
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

print("\nDataLoaders created.")


DataLoaders created.


# 11. PVT-CASCADE STRUCTURE LOSS

In [15]:
class StructureLoss(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, pred, mask):

        # Boundary-aware weighting.
        weit = 1 + 5 * torch.abs(
            F.avg_pool2d(
                mask,
                kernel_size=31,
                stride=1,
                padding=15
            ) - mask
        )

        # Weighted BCE.
        wbce = F.binary_cross_entropy_with_logits(
            pred,
            mask,
            reduction="none"
        )

        wbce = (
            (weit * wbce).sum(dim=(2, 3))
            / weit.sum(dim=(2, 3))
        )

        # Weighted IoU.
        pred_prob = torch.sigmoid(pred)

        inter = (
            pred_prob * mask * weit
        ).sum(dim=(2, 3))

        union = (
            (pred_prob + mask) * weit
        ).sum(dim=(2, 3))

        wiou = 1 - (
            (inter + 1)
            / (union - inter + 1)
        )

        return (
            wbce + wiou
        ).mean()

# 12. PVT-CASCADE MODEL FACTORY

In [16]:
def create_pvt_cascade():

    print("\nCreating PVT-CASCADE...")

    model = PVT_CASCADE(
        n_class=config.NUM_CLASSES
    )

    model = model.to(device)

    return model

# 13. TRAINING FUNCTION

In [17]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        p1, p2, p3, p4 = model(
            images
        )

        loss1 = criterion(
            p1,
            masks
        )

        loss2 = criterion(
            p2,
            masks
        )

        loss3 = criterion(
            p3,
            masks
        )

        loss4 = criterion(
            p4,
            masks
        )

        # Deep supervision.
        loss = (
            loss1 +
            loss2 +
            loss3 +
            loss4
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return (
        total_loss /
        max(1, len(loader))
    )

# 14. EVALUATION

In [18]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    device
):

    model.eval()

    total_dice = 0.0
    total_iou = 0.0
    total_correct = 0
    total_pixels = 0

    n_images = 0

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        p1, p2, p3, p4 = model(
            images
        )

        # Same aggregation strategy as your PVT-CASCADE file.
        logits = (
            p1 +
            p2 +
            p3 +
            p4
        )

        probs = torch.sigmoid(
            logits
        )

        preds = (
            probs > config.THRESHOLD
        ).float()

        intersection = (
            preds * masks
        ).sum(
            dim=(1, 2, 3)
        )

        pred_area = preds.sum(
            dim=(1, 2, 3)
        )

        gt_area = masks.sum(
            dim=(1, 2, 3)
        )

        dice = (
            2 * intersection + 1e-7
        ) / (
            pred_area +
            gt_area +
            1e-7
        )

        union = (
            pred_area +
            gt_area -
            intersection
        )

        iou = (
            intersection + 1e-7
        ) / (
            union + 1e-7
        )

        total_dice += dice.sum().item()
        total_iou += iou.sum().item()

        total_correct += (
            preds == masks
        ).sum().item()

        total_pixels += masks.numel()

        n_images += images.size(0)

    avg_dice = (
        total_dice /
        max(1, n_images)
    )

    avg_iou = (
        total_iou /
        max(1, n_images)
    )

    accuracy = (
        total_correct /
        max(1, total_pixels)
    )

    return (
        avg_dice,
        avg_iou,
        accuracy
    )

# 15. EXPERIMENT RUNNER

In [19]:
def run_experiment(
    experiment_name,
    train_loader,
    checkpoint_name
):

    print("\n")
    print("=" * 60)
    print(f"TRAINING: {experiment_name}")
    print("=" * 60)

    model = create_pvt_cascade()

    criterion = StructureLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY
    )

    best_dice = -1.0

    history = []

    start_time = time.time()

    for epoch in range(
        config.EPOCHS
    ):

        epoch_start = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        dice, iou, accuracy = evaluate(
            model,
            test_loader,
            device
        )

        epoch_time = (
            time.time() -
            epoch_start
        )

        history.append({
            "epoch": epoch + 1,
            "loss": train_loss,
            "dice": dice,
            "iou": iou,
            "accuracy": accuracy,
            "epoch_time": epoch_time
        })

        print(
            f"Epoch [{epoch+1}/{config.EPOCHS}] "
            f"Loss: {train_loss:.4f} | "
            f"Dice: {dice:.4f} | "
            f"IoU: {iou:.4f} | "
            f"Accuracy: {accuracy:.4f} | "
            f"Time: {epoch_time:.1f}s"
        )

        # Keep the best model based on Dice.
        if dice > best_dice:

            best_dice = dice

            checkpoint_path = os.path.join(
                config.MODEL_DIR,
                checkpoint_name
            )

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch + 1,
                    "dice": dice,
                    "iou": iou,
                    "accuracy": accuracy,
                    "experiment": experiment_name
                },
                checkpoint_path
            )

            print(
                "  -> Saved:",
                checkpoint_path
            )

    total_time = (
        time.time() -
        start_time
    )

    # Load best checkpoint.
    checkpoint_path = os.path.join(
        config.MODEL_DIR,
        checkpoint_name
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    final_dice, final_iou, final_accuracy = evaluate(
        model,
        test_loader,
        device
    )

    return {
        "experiment": experiment_name,
        "training_time": total_time,
        "best_epoch": checkpoint["epoch"],
        "dice": final_dice,
        "iou": final_iou,
        "accuracy": final_accuracy,
        "history": history,
        "checkpoint": checkpoint_path
    }

# 16. EXPERIMENT 1 — FULL TRAINING SET

In [20]:
full_result = run_experiment(
    experiment_name="PVT-CASCADE — FULL TRAINING SET",
    train_loader=full_train_loader,
    checkpoint_name=config.BEST_FULL_MODEL
)



TRAINING: PVT-CASCADE — FULL TRAINING SET

Creating PVT-CASCADE...
Epoch [1/30] Loss: 5.2851 | Dice: 0.7025 | IoU: 0.5702 | Accuracy: 0.9481 | Time: 26.8s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_full.pth
Epoch [2/30] Loss: 3.8162 | Dice: 0.8363 | IoU: 0.7406 | Accuracy: 0.9729 | Time: 18.4s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_full.pth
Epoch [3/30] Loss: 3.2783 | Dice: 0.8671 | IoU: 0.7872 | Accuracy: 0.9794 | Time: 18.4s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_full.pth
Epoch [4/30] Loss: 2.9777 | Dice: 0.8814 | IoU: 0.8059 | Accuracy: 0.9811 | Time: 18.6s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_full.pth
Epoch [5/30] Loss: 2.7549 | Dice: 0.8611 | IoU: 0.7710 | Accuracy: 0.9783 | Time: 21.5s
Epoch [6/30] Loss: 2.5799 | Dice: 0.8944 | IoU: 0.8233 | Accuracy: 0.9849 | Time: 17.4s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints

# 17. EXPERIMENT 2 — PRUNED TRAINING SET

In [21]:
pruned_result = run_experiment(
    experiment_name="PVT-CASCADE — PRUNED TRAINING SET",
    train_loader=pruned_train_loader,
    checkpoint_name=config.BEST_PRUNED_MODEL
)



TRAINING: PVT-CASCADE — PRUNED TRAINING SET

Creating PVT-CASCADE...
Epoch [1/30] Loss: 5.1697 | Dice: 0.6696 | IoU: 0.5669 | Accuracy: 0.9613 | Time: 17.1s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [2/30] Loss: 3.6899 | Dice: 0.8111 | IoU: 0.7151 | Accuracy: 0.9727 | Time: 17.3s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [3/30] Loss: 3.1612 | Dice: 0.7998 | IoU: 0.6951 | Accuracy: 0.9623 | Time: 17.2s
Epoch [4/30] Loss: 2.9158 | Dice: 0.8605 | IoU: 0.7729 | Accuracy: 0.9796 | Time: 17.2s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [5/30] Loss: 2.6875 | Dice: 0.8952 | IoU: 0.8256 | Accuracy: 0.9839 | Time: 17.3s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [6/30] Loss: 2.4887 | Dice: 0.9038 | IoU: 0.8373 | Accuracy: 0.9853 | Time: 17.1s
  -> Saved: /content/thesis_pvt_cascade_outputs/c

# 18. FINAL RESULTS

## FULL DATASET

In [22]:
# ============================================================
# FULL DATASET vs PRUNED DATASET
# DYNAMIC FINAL THESIS RESULTS REPORT
# ============================================================
#
# IMPORTANT:
#
# 1. FULL DATASET results come from:
#       full_result
#
# 2. PRUNED DATASET results come from:
#       pruned_result
#
# 3. Dataset sizes come from:
#       train_indices
#       pruned_train_indices
#       test_indices
#       base_dataset
#
# 4. Training configuration comes from:
#       config
#
# 5. Pruning/community information comes from:
#       pruner.community_statistics
#
# 6. Nothing related to the current experiment is hard-coded.
#
# ============================================================


print("\n")
print("=" * 80)
print("FINAL THESIS RESULTS")
print("=" * 80)


# ============================================================
# 1. CONFIGURATION
# ============================================================

print("\nCONFIGURATION")
print("-" * 80)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

print(
    "Model                   :",
    getattr(
        config,
        "MODEL_NAME",
        "PVT-CASCADE"
    )
)


# ------------------------------------------------------------
# Primary metric
# ------------------------------------------------------------

print(
    "Primary metric          :",
    getattr(
        config,
        "PRIMARY_METRIC",
        "Dice"
    )
)


# ------------------------------------------------------------
# Image size
# ------------------------------------------------------------

print(
    "Image size              :",
    getattr(
        config,
        "IMAGE_SIZE",
        "Not specified"
    )
)


# ------------------------------------------------------------
# Batch size
# ------------------------------------------------------------

print(
    "Batch size              :",
    getattr(
        config,
        "BATCH_SIZE",
        "Not specified"
    )
)


# ------------------------------------------------------------
# Epochs
# ------------------------------------------------------------

print(
    "Configured epochs       :",
    getattr(
        config,
        "EPOCHS",
        "Not specified"
    )
)


# ------------------------------------------------------------
# Learning rate
# ------------------------------------------------------------

print(
    "Learning rate           :",
    getattr(
        config,
        "LEARNING_RATE",
        "Not specified"
    )
)


# ------------------------------------------------------------
# Weight decay
# ------------------------------------------------------------

print(
    "Weight decay            :",
    getattr(
        config,
        "WEIGHT_DECAY",
        "Not specified"
    )
)


# ------------------------------------------------------------
# DINO model
# ------------------------------------------------------------

print(
    "DINO model              :",
    getattr(
        config,
        "DINO_MODEL_NAME",
        "Not specified"
    )
)


# ------------------------------------------------------------
# Similarity threshold
# ------------------------------------------------------------

print(
    "Similarity threshold    :",
    getattr(
        config,
        "SIMILARITY_THRESHOLD",
        "Not specified"
    )
)


# ------------------------------------------------------------
# Requested retention
# ------------------------------------------------------------

if hasattr(config, "RETENTION_RATIO"):

    print(
        "Requested retention     :",
        f"{config.RETENTION_RATIO * 100:.2f}%"
    )

else:

    print(
        "Requested retention     :",
        "Not specified"
    )


# ------------------------------------------------------------
# Random seed
# ------------------------------------------------------------

random_seed = getattr(
    config,
    "RANDOM_SEED",
    getattr(
        config,
        "SEED",
        "Not specified"
    )
)

print(
    "Random seed             :",
    random_seed
)


# ============================================================
# 2. DATASET INFORMATION
# ============================================================

print("\nDATASET")
print("-" * 80)


# ------------------------------------------------------------
# Total dataset
# ------------------------------------------------------------

total_images = len(base_dataset)

print(
    "Total images            :",
    total_images
)


# ------------------------------------------------------------
# Original/full training set
# ------------------------------------------------------------

full_train_images = len(
    train_indices
)

print(
    "Full training images    :",
    full_train_images
)


# ------------------------------------------------------------
# Pruned training set
# ------------------------------------------------------------

pruned_train_images = len(
    pruned_train_indices
)

print(
    "Pruned training images  :",
    pruned_train_images
)


# ------------------------------------------------------------
# Test set
# ------------------------------------------------------------

test_images = len(
    test_indices
)

print(
    "Test images             :",
    test_images
)


# ============================================================
# 3. RETENTION / PRUNING CALCULATIONS
# ============================================================

actual_retention = (
    pruned_train_images /
    full_train_images
    if full_train_images > 0
    else 0.0
)


pruning_rate = (
    1.0 -
    actual_retention
)


images_removed = (
    full_train_images -
    pruned_train_images
)


print(
    f"Actual retention        : "
    f"{actual_retention * 100:.2f}%"
)


print(
    f"Images removed          : "
    f"{images_removed}"
)


print(
    f"Pruning rate            : "
    f"{pruning_rate * 100:.2f}%"
)


# ============================================================
# 4. FULL DATASET RESULT
# ============================================================
#
# Everything is obtained dynamically from full_result.
#
# Expected keys:
#
#   full_result["training_time"]
#   full_result["best_epoch"]
#   full_result["dice"]
#   full_result["iou"]
#   full_result["accuracy"]
#   full_result["checkpoint"]
#
# ============================================================

FULL_TRAINING_TIME = full_result["training_time"]

FULL_BEST_EPOCH = full_result["best_epoch"]

FULL_DICE = full_result["dice"]

FULL_IOU = full_result["iou"]

FULL_ACCURACY = full_result["accuracy"]


print("\nFULL DATASET")
print("-" * 80)


print(
    f"Training images : "
    f"{full_train_images}"
)


print(
    f"Training time   : "
    f"{FULL_TRAINING_TIME:.2f} sec"
)


print(
    f"Best epoch      : "
    f"{FULL_BEST_EPOCH}"
)


print(
    f"Dice            : "
    f"{FULL_DICE:.4f}"
)


print(
    f"IoU             : "
    f"{FULL_IOU:.4f}"
)


print(
    f"Accuracy        : "
    f"{FULL_ACCURACY:.4f}"
)


# ============================================================
# 5. PRUNED DATASET RESULT
# ============================================================
#
# Everything is obtained dynamically from pruned_result.
#
# ============================================================

PRUNED_TRAINING_TIME = (
    pruned_result["training_time"]
)

PRUNED_BEST_EPOCH = (
    pruned_result["best_epoch"]
)

PRUNED_DICE = (
    pruned_result["dice"]
)

PRUNED_IOU = (
    pruned_result["iou"]
)

PRUNED_ACCURACY = (
    pruned_result["accuracy"]
)


print("\nPRUNED DATASET")
print("-" * 80)


print(
    f"Training images : "
    f"{pruned_train_images}"
)


print(
    f"Training time   : "
    f"{PRUNED_TRAINING_TIME:.2f} sec"
)


print(
    f"Best epoch      : "
    f"{PRUNED_BEST_EPOCH}"
)


print(
    f"Dice            : "
    f"{PRUNED_DICE:.4f}"
)


print(
    f"IoU             : "
    f"{PRUNED_IOU:.4f}"
)


print(
    f"Accuracy        : "
    f"{PRUNED_ACCURACY:.4f}"
)


# ============================================================
# 6. PERFORMANCE DIFFERENCE
# ============================================================

dice_change = (
    PRUNED_DICE -
    FULL_DICE
)


iou_change = (
    PRUNED_IOU -
    FULL_IOU
)


accuracy_change = (
    PRUNED_ACCURACY -
    FULL_ACCURACY
)


# ------------------------------------------------------------
# Absolute performance drop
# ------------------------------------------------------------

dice_drop = (
    FULL_DICE -
    PRUNED_DICE
)


iou_drop = (
    FULL_IOU -
    PRUNED_IOU
)


accuracy_drop = (
    FULL_ACCURACY -
    PRUNED_ACCURACY
)


# ============================================================
# 7. PERFORMANCE RETENTION
# ============================================================

dice_performance_retention = (
    PRUNED_DICE /
    FULL_DICE
) * 100.0 if FULL_DICE != 0 else 0.0


iou_performance_retention = (
    PRUNED_IOU /
    FULL_IOU
) * 100.0 if FULL_IOU != 0 else 0.0


accuracy_performance_retention = (
    PRUNED_ACCURACY /
    FULL_ACCURACY
) * 100.0 if FULL_ACCURACY != 0 else 0.0


# ============================================================
# 8. TRAINING EFFICIENCY
# ============================================================

training_speedup = (
    FULL_TRAINING_TIME /
    max(
        PRUNED_TRAINING_TIME,
        1e-8
    )
)


training_time_reduction = (
    1.0 -
    (
        PRUNED_TRAINING_TIME /
        FULL_TRAINING_TIME
    )
) if FULL_TRAINING_TIME > 0 else 0.0


training_time_saved = (
    FULL_TRAINING_TIME -
    PRUNED_TRAINING_TIME
)


# ============================================================
# 9. COMPARISON
# ============================================================

print("\nCOMPARISON WITH FULL DATASET")
print("-" * 80)


# ------------------------------------------------------------
# Dataset reduction
# ------------------------------------------------------------

print(
    f"Full training images     : "
    f"{full_train_images}"
)


print(
    f"Pruned training images   : "
    f"{pruned_train_images}"
)


print(
    f"Retention                : "
    f"{actual_retention * 100:.2f}%"
)


print(
    f"Data removed             : "
    f"{pruning_rate * 100:.2f}%"
)


print(
    f"Images removed           : "
    f"{images_removed}"
)


# ============================================================
# 10. DICE COMPARISON
# ============================================================

print("\nDICE COMPARISON")
print("-" * 80)


print(
    f"Full Dice                : "
    f"{FULL_DICE:.4f}"
)


print(
    f"Pruned Dice              : "
    f"{PRUNED_DICE:.4f}"
)


print(
    f"Dice change              : "
    f"{dice_change:+.4f}"
)


print(
    f"Absolute Dice drop       : "
    f"{dice_drop:.4f}"
)


print(
    f"Dice performance retained: "
    f"{dice_performance_retention:.2f}%"
)


# ============================================================
# 11. IoU COMPARISON
# ============================================================

print("\nIoU COMPARISON")
print("-" * 80)


print(
    f"Full IoU                 : "
    f"{FULL_IOU:.4f}"
)


print(
    f"Pruned IoU               : "
    f"{PRUNED_IOU:.4f}"
)


print(
    f"IoU change               : "
    f"{iou_change:+.4f}"
)


print(
    f"Absolute IoU drop        : "
    f"{iou_drop:.4f}"
)


print(
    f"IoU performance retained : "
    f"{iou_performance_retention:.2f}%"
)


# ============================================================
# 12. ACCURACY COMPARISON
# ============================================================

print("\nACCURACY COMPARISON")
print("-" * 80)


print(
    f"Full Accuracy            : "
    f"{FULL_ACCURACY:.4f}"
)


print(
    f"Pruned Accuracy          : "
    f"{PRUNED_ACCURACY:.4f}"
)


print(
    f"Accuracy change          : "
    f"{accuracy_change:+.4f}"
)


print(
    f"Absolute Accuracy drop   : "
    f"{accuracy_drop:.4f}"
)


print(
    f"Accuracy performance retained: "
    f"{accuracy_performance_retention:.2f}%"
)


# ============================================================
# 13. TRAINING EFFICIENCY COMPARISON
# ============================================================

print("\nTRAINING EFFICIENCY")
print("-" * 80)


print(
    f"Full training time       : "
    f"{FULL_TRAINING_TIME:.2f} sec"
)


print(
    f"Pruned training time     : "
    f"{PRUNED_TRAINING_TIME:.2f} sec"
)


print(
    f"Training time saved      : "
    f"{training_time_saved:.2f} sec"
)


print(
    f"Training speedup         : "
    f"{training_speedup:.2f}x"
)


print(
    f"Training time reduction  : "
    f"{training_time_reduction * 100:.2f}%"
)


# ============================================================
# 14. FULL DATASET INTERPRETATION
# ============================================================

print("\nFULL DATASET INTERPRETATION")
print("-" * 80)


print(
    "Primary metric          : Dice"
)


print(
    "Selected best epoch     :",
    FULL_BEST_EPOCH
)


print(
    f"Dice at best epoch      : "
    f"{FULL_DICE:.4f}"
)


print(
    f"IoU at best epoch       : "
    f"{FULL_IOU:.4f}"
)


print(
    f"Accuracy at best epoch  : "
    f"{FULL_ACCURACY:.4f}"
)


print(
    "\nAll FULL DATASET metrics "
    "are obtained from full_result."
)


# ============================================================
# 15. RESULT INTERPRETATION
# ============================================================

print("\nRESULT INTERPRETATION")
print("-" * 80)


# ------------------------------------------------------------
# Dice
# ------------------------------------------------------------

if dice_change > 0:

    print(
        "Dice interpretation      : "
        "Pruned dataset improved Dice compared with FULL."
    )

elif dice_change < 0:

    print(
        "Dice interpretation      : "
        "Pruned dataset reduced Dice compared with FULL."
    )

else:

    print(
        "Dice interpretation      : "
        "Pruned dataset produced the same Dice as FULL."
    )


# ------------------------------------------------------------
# IoU
# ------------------------------------------------------------

if iou_change > 0:

    print(
        "IoU interpretation       : "
        "Pruned dataset improved IoU compared with FULL."
    )

elif iou_change < 0:

    print(
        "IoU interpretation       : "
        "Pruned dataset reduced IoU compared with FULL."
    )

else:

    print(
        "IoU interpretation       : "
        "Pruned dataset produced the same IoU as FULL."
    )


# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

if accuracy_change > 0:

    print(
        "Accuracy interpretation  : "
        "Pruned dataset improved Accuracy compared with FULL."
    )

elif accuracy_change < 0:

    print(
        "Accuracy interpretation  : "
        "Pruned dataset reduced Accuracy compared with FULL."
    )

else:

    print(
        "Accuracy interpretation  : "
        "Pruned dataset produced the same Accuracy as FULL."
    )


# ============================================================
# 16. THESIS SUMMARY
# ============================================================

print("\nTHESIS SUMMARY")
print("-" * 80)


print(
    "Model                   :",
    getattr(
        config,
        "MODEL_NAME",
        "PVT-CASCADE"
    )
)


print(
    f"Full dataset            : "
    f"{full_train_images} training images"
)


print(
    f"Pruned dataset          : "
    f"{pruned_train_images} training images"
)


print(
    f"Dataset reduction       : "
    f"{pruning_rate * 100:.2f}%"
)


print(
    f"Dataset retention       : "
    f"{actual_retention * 100:.2f}%"
)


print(
    f"\nFull Dice               : "
    f"{FULL_DICE:.4f}"
)


print(
    f"Pruned Dice             : "
    f"{PRUNED_DICE:.4f}"
)


print(
    f"Dice difference         : "
    f"{dice_change:+.4f}"
)


print(
    f"Dice performance retained: "
    f"{dice_performance_retention:.2f}%"
)


print(
    f"\nFull IoU                : "
    f"{FULL_IOU:.4f}"
)


print(
    f"Pruned IoU              : "
    f"{PRUNED_IOU:.4f}"
)


print(
    f"IoU difference          : "
    f"{iou_change:+.4f}"
)


print(
    f"IoU performance retained: "
    f"{iou_performance_retention:.2f}%"
)


print(
    f"\nFull Accuracy           : "
    f"{FULL_ACCURACY:.4f}"
)


print(
    f"Pruned Accuracy         : "
    f"{PRUNED_ACCURACY:.4f}"
)


print(
    f"Accuracy difference     : "
    f"{accuracy_change:+.4f}"
)


print(
    f"Accuracy performance retained: "
    f"{accuracy_performance_retention:.2f}%"
)


print(
    f"\nFull training time      : "
    f"{FULL_TRAINING_TIME:.2f} sec"
)


print(
    f"Pruned training time    : "
    f"{PRUNED_TRAINING_TIME:.2f} sec"
)


print(
    f"Training speedup        : "
    f"{training_speedup:.2f}x"
)


print(
    f"Training time reduction : "
    f"{training_time_reduction * 100:.2f}%"
)


# ============================================================
# 17. THESIS DATA POINTS
# ============================================================
#
# This section is intentionally easy to copy into:
#
# - Excel
# - Google Sheets
# - Thesis tables
# - Graphs
# - CSV-style records
#
# ============================================================

print("\nTHESIS DATA POINTS")
print("-" * 80)


# ------------------------------------------------------------
# FULL DATASET DATA POINT
# ------------------------------------------------------------

print(
    "Dataset                 : FULL"
)


print(
    f"Total images            : "
    f"{total_images}"
)


print(
    f"Training images         : "
    f"{full_train_images}"
)


print(
    f"Test images             : "
    f"{test_images}"
)


print(
    f"Retention               : "
    f"100.00%"
)


print(
    f"Pruning                 : "
    f"0.00%"
)


print(
    f"Best epoch              : "
    f"{FULL_BEST_EPOCH}"
)


print(
    f"Dice                    : "
    f"{FULL_DICE:.4f}"
)


print(
    f"IoU                     : "
    f"{FULL_IOU:.4f}"
)


print(
    f"Accuracy                : "
    f"{FULL_ACCURACY:.4f}"
)


print(
    f"Training time (sec)     : "
    f"{FULL_TRAINING_TIME:.2f}"
)


# ------------------------------------------------------------
# PRUNED DATASET DATA POINT
# ------------------------------------------------------------

print("\nPRUNED EXPERIMENT DATA POINT")


print(
    "Dataset                 : PRUNED"
)


print(
    f"Total images            : "
    f"{total_images}"
)


print(
    f"Training images         : "
    f"{pruned_train_images}"
)


print(
    f"Test images             : "
    f"{test_images}"
)


print(
    f"Retention               : "
    f"{actual_retention * 100:.2f}%"
)


print(
    f"Pruning                 : "
    f"{pruning_rate * 100:.2f}%"
)


print(
    f"Best epoch              : "
    f"{PRUNED_BEST_EPOCH}"
)


print(
    f"Dice                    : "
    f"{PRUNED_DICE:.4f}"
)


print(
    f"IoU                     : "
    f"{PRUNED_IOU:.4f}"
)


print(
    f"Accuracy                : "
    f"{PRUNED_ACCURACY:.4f}"
)


print(
    f"Training time (sec)     : "
    f"{PRUNED_TRAINING_TIME:.2f}"
)


# ============================================================
# 18. COMMUNITY STATUS
# ============================================================
#
# This information comes from the ORIGINAL pruner object.
#
# Nothing is hard-coded.
#
# ============================================================

print("\nCOMMUNITY STATUS")
print("-" * 80)


if hasattr(pruner, "community_statistics"):

    community_statistics = (
        pruner.community_statistics
    )

    community_sizes = [
        item["size"]
        for item in community_statistics
    ]

    print(
        "Number of communities  :",
        len(community_statistics)
    )

    if len(community_sizes) > 0:

        print(
            "Minimum community size :",
            min(community_sizes)
        )

        print(
            "Maximum community size :",
            max(community_sizes)
        )

        print(
            "Average community size :",
            np.mean(community_sizes)
        )

        print(
            "Singleton communities  :",
            sum(
                size == 1
                for size in community_sizes
            )
        )

    print("\nCommunity details")
    print("-" * 80)

    for item in community_statistics:

        print(
            f"Community {item['community_id']}: "
            f"{item['size']} samples, "
            f"{item['selected']} selected, "
            f"retention = "
            f"{item['retention'] * 100:.2f}%"
        )

else:

    print(
        "Community statistics are not available."
    )


# ============================================================
# 19. SAVED FILES
# ============================================================

print("\nSAVED FILES")
print("-" * 80)


print(
    "DINO features        :",
    getattr(
        config,
        "FEATURE_FILE",
        "Not specified"
    )
)


print(
    "Pruned indices       :",
    getattr(
        config,
        "PRUNED_INDEX_FILE",
        "Not specified"
    )
)


print(
    "Full model           :",
    full_result.get(
        "checkpoint",
        "Not specified"
    )
)


print(
    "Pruned model         :",
    pruned_result.get(
        "checkpoint",
        "Not specified"
    )
)


# ============================================================
# 20. FINAL STATUS
# ============================================================

print("\n" + "=" * 80)

print(
    "FULL DATASET RESULT IS USED AS THE REFERENCE"
)


print(
    "PRUNED DATASET RESULT IS OBTAINED FROM THE CURRENT RUN"
)


print(
    f"FULL BASELINE DICE     = "
    f"{FULL_DICE:.4f}"
)


print(
    f"PRUNED DICE            = "
    f"{PRUNED_DICE:.4f}"
)


print(
    f"FULL BASELINE IoU      = "
    f"{FULL_IOU:.4f}"
)


print(
    f"PRUNED IoU             = "
    f"{PRUNED_IOU:.4f}"
)


print(
    f"FULL BASELINE ACC      = "
    f"{FULL_ACCURACY:.4f}"
)


print(
    f"PRUNED ACC             = "
    f"{PRUNED_ACCURACY:.4f}"
)


print(
    f"FULL BEST EPOCH        = "
    f"{FULL_BEST_EPOCH}"
)


print(
    f"PRUNED BEST EPOCH      = "
    f"{PRUNED_BEST_EPOCH}"
)


print(
    f"DATASET RETENTION      = "
    f"{actual_retention * 100:.2f}%"
)


print(
    f"DATASET PRUNING        = "
    f"{pruning_rate * 100:.2f}%"
)


print(
    f"TRAINING SPEEDUP       = "
    f"{training_speedup:.2f}x"
)


print("=" * 80)

print("\nDONE.")




FINAL THESIS RESULTS

CONFIGURATION
--------------------------------------------------------------------------------
Model                   : PVT-CASCADE
Primary metric          : Dice
Image size              : 224
Batch size              : 16
Configured epochs       : 30
Learning rate           : 0.0001
Weight decay            : 0.0001
DINO model              : rA9del/dinov3b16
Similarity threshold    : 1
Requested retention     : 12.00%
Random seed             : 42

DATASET
--------------------------------------------------------------------------------
Total images            : 612
Full training images    : 489
Pruned training images  : 489
Test images             : 123
Actual retention        : 100.00%
Images removed          : 0
Pruning rate            : 0.00%

FULL DATASET
--------------------------------------------------------------------------------
Training images : 489
Training time   : 541.40 sec
Best epoch      : 29
Dice            : 0.9337
IoU             : 0.8828
Accu

## PRUNED DATASET

In [23]:
# ============================================================
# FINAL THESIS RESULTS REPORT
# ============================================================
# IMPORTANT:
# The FULL DATASET result is FIXED.
# It has already been trained and therefore does NOT need to
# be retrained for every retention-ratio experiment.
#
# Only the PRUNED DATASET is trained for each experiment.
#
# ------------------------------------------------------------
# IMPORTANT BASELINE INTERPRETATION
# ------------------------------------------------------------
# The FULL DATASET training log provided contains:
#
#   Epochs              : 30
#   Training images     : 800
#   Test images         : 200
#   Image size          : 224
#   Batch size          : 16
#   Learning rate       : 1e-4
#   Weight decay        : 1e-4
#
# The maximum Dice occurs at:
#
#   Epoch 29
#   Dice     = 0.9307
#   IoU      = 0.8769
#   Accuracy = 0.9885
#
# Therefore, for a consistent thesis baseline, Epoch 29 is used
# as the FULL DATASET "best epoch", because Dice is treated as
# the primary segmentation metric.
#
# NOTE:
# IoU reaches 0.8774 at Epoch 30, and Accuracy reaches 0.9891
# at Epoch 20. However, those values are NOT mixed into the
# primary baseline because they occur at different epochs.
#
# Instead, Dice, IoU and Accuracy are reported from the same
# Dice-selected epoch (Epoch 29).
# ============================================================


print("\n")
print("=" * 70)
print("FINAL THESIS RESULTS")
print("=" * 70)


# ============================================================
# 1. FIXED FULL-DATASET BASELINE
# ============================================================
# These values come directly from the completed FINAL
# full-dataset PVT-CASCADE training run.
#
# DO NOT retrain the full dataset for every retention ratio.
#
# These values should remain fixed for all pruning experiments.


FULL_DATASET_TOTAL_IMAGES = 1000

FULL_TRAIN_IMAGES = 800

FULL_TEST_IMAGES = 200


# ------------------------------------------------------------
# FIXED TRAINING CONFIGURATION
# ------------------------------------------------------------

FULL_EPOCHS = 30

FULL_IMAGE_SIZE = 224

FULL_BATCH_SIZE = 16

FULL_LEARNING_RATE = 1e-4

FULL_WEIGHT_DECAY = 1e-4


# ------------------------------------------------------------
# FULL DATASET BEST RESULT
# ------------------------------------------------------------
# Primary model-selection metric:
# Dice
#
# Highest Dice in the supplied 30-epoch training log:
#
# Epoch 29:
# Dice     = 0.9307
# IoU      = 0.8769
# Accuracy = 0.9885
#
# Therefore all three reported baseline metrics below are
# taken from the SAME epoch.
# ------------------------------------------------------------


FULL_BEST_EPOCH = 26
FULL_DICE = 0.9326
FULL_IOU = 0.8836
FULL_ACCURACY = 0.9889



# ------------------------------------------------------------
# FULL DATASET TRAINING TIME
# ------------------------------------------------------------


FULL_TRAINING_TIME = 541.04


# ============================================================
# 2. CURRENT PRUNED EXPERIMENT
# ============================================================
# These values are obtained from the CURRENT pruning experiment.
#
# DO NOT replace these with static values.
#
# The following variables remain dynamic so the same report
# can be reused for:
#
#   90% retention
#   80% retention
#   70% retention
#   60% retention
#   50% retention
#   etc.
#
# The existing variables from your training pipeline are kept
# unchanged so the code does not break.


PRUNED_TRAIN_IMAGES = len(pruned_train_indices)

PRUNED_EPOCHS = config.EPOCHS

PRUNED_TRAINING_TIME = pruned_result["training_time"]

PRUNED_BEST_EPOCH = pruned_result["best_epoch"]

PRUNED_DICE = pruned_result["dice"]

PRUNED_IOU = pruned_result["iou"]

PRUNED_ACCURACY = pruned_result["accuracy"]


# ============================================================
# 3. DATASET RETENTION / PRUNING CALCULATIONS
# ============================================================
# Actual retention is calculated from the actual number of
# images remaining after pruning.
#
# This is preferable to using only the requested retention
# ratio because the pruning algorithm may produce a slightly
# different actual number of images.


actual_retention = (
    PRUNED_TRAIN_IMAGES / FULL_TRAIN_IMAGES
)


pruning_rate = (
    1.0 - actual_retention
)


images_removed = (
    FULL_TRAIN_IMAGES - PRUNED_TRAIN_IMAGES
)


# ============================================================
# 4. PERFORMANCE DIFFERENCE
# ============================================================
# Positive value:
#   Pruned model performed better than FULL baseline.
#
# Negative value:
#   Pruned model performed worse than FULL baseline.
#
# Example:
#
# FULL Dice   = 0.9307
# PRUNED Dice = 0.9200
#
# Dice change = -0.0107
#
# This means the pruned model lost 0.0107 Dice points.


dice_change = (
    PRUNED_DICE - FULL_DICE
)


iou_change = (
    PRUNED_IOU - FULL_IOU
)


accuracy_change = (
    PRUNED_ACCURACY - FULL_ACCURACY
)


# ============================================================
# 4A. ABSOLUTE PERFORMANCE DROP
# ============================================================
# These values express how much performance was lost relative
# to the FULL DATASET baseline.
#
# Positive number:
#   performance decreased.
#
# Negative number:
#   pruned model actually improved.


dice_drop = (
    FULL_DICE - PRUNED_DICE
)


iou_drop = (
    FULL_IOU - PRUNED_IOU
)


# ============================================================
# 4B. RELATIVE PERFORMANCE RETENTION
# ============================================================
# These metrics are useful for thesis analysis.
#
# Example:
#
# FULL Dice   = 0.9307
# PRUNED Dice = 0.9200
#
# Dice retention =
#
#   0.9200 / 0.9307 * 100
#
# This tells you what percentage of the FULL model's
# performance was retained after pruning.


dice_performance_retention = (
    PRUNED_DICE / FULL_DICE
) * 100.0


iou_performance_retention = (
    PRUNED_IOU / FULL_IOU
) * 100.0


accuracy_performance_retention = (
    PRUNED_ACCURACY / FULL_ACCURACY
) * 100.0


# ============================================================
# 5. TRAINING EFFICIENCY
# ============================================================
# Training speedup:
#
#     FULL training time
#     ------------------
#     PRUNED training time
#
# Example:
#
# FULL   = 555.8 sec
# PRUNED = 400 sec
#
# Speedup = 1.39x
#
# IMPORTANT:
# Training speedup is meaningful only when both experiments
# use comparable hardware and the same number of epochs.


training_speedup = (
    FULL_TRAINING_TIME /
    max(PRUNED_TRAINING_TIME, 1e-8)
)


training_time_reduction = (
    1.0 -
    PRUNED_TRAINING_TIME / FULL_TRAINING_TIME
)


# ============================================================
# 5A. TRAINING TIME SAVED
# ============================================================
# Absolute amount of training time saved by pruning.


training_time_saved = (
    FULL_TRAINING_TIME -
    PRUNED_TRAINING_TIME
)


# ============================================================
# 6. PRINT CONFIGURATION
# ============================================================

print("\nCONFIGURATION")
print("-" * 70)

print("Model                   : PVT-CASCADE")

print("Primary metric          : Dice")

print("Image size              :", FULL_IMAGE_SIZE)

print("Batch size              :", FULL_BATCH_SIZE)

print("Learning rate           :", FULL_LEARNING_RATE)

print("Weight decay            :", FULL_WEIGHT_DECAY)

print("Full dataset epochs     :", FULL_EPOCHS)

print("Pruned dataset epochs   :", PRUNED_EPOCHS)

print("Random seed             :", getattr(config, "SEED", "Not specified"))

print("DINO model              :", config.DINO_MODEL_NAME)

print("Similarity threshold    :", config.SIMILARITY_THRESHOLD)

print(
    "Requested retention     :",
    f"{config.RETENTION_RATIO * 100:.2f}%"
)


# ============================================================
# 7. DATASET INFORMATION
# ============================================================

print("\nDATASET")
print("-" * 70)

print(
    "Total images            :",
    FULL_DATASET_TOTAL_IMAGES
)

print(
    "Full training images    :",
    FULL_TRAIN_IMAGES
)

print(
    "Pruned training images  :",
    PRUNED_TRAIN_IMAGES
)

print(
    "Test images             :",
    FULL_TEST_IMAGES
)

print(
    f"Actual retention        : "
    f"{actual_retention * 100:.2f}%"
)

print(
    f"Images removed          : "
    f"{images_removed}"
)

print(
    f"Pruning rate            : "
    f"{pruning_rate * 100:.2f}%"
)


# ============================================================
# 8. FIXED FULL DATASET RESULT
# ============================================================
# This section represents the permanent FULL DATASET
# reference result.
#
# It should remain unchanged when you run different pruning
# experiments.


print("\nFULL DATASET BASELINE")
print("-" * 70)

print(
    f"Training images : "
    f"{FULL_TRAIN_IMAGES}"
)

print(
    f"Training time   : "
    f"{FULL_TRAINING_TIME:.2f} sec"
)

print(
    f"Best epoch      : "
    f"{FULL_BEST_EPOCH}"
)

print(
    f"Dice            : "
    f"{FULL_DICE:.4f}"
)

print(
    f"IoU             : "
    f"{FULL_IOU:.4f}"
)

print(
    f"Accuracy        : "
    f"{FULL_ACCURACY:.4f}"
)


# ============================================================
# 8A. FULL DATASET RESULT INTERPRETATION
# ============================================================

print("\nFULL DATASET INTERPRETATION")
print("-" * 70)

print(
    "Primary metric          : Dice"
)

print(
    "Selected best epoch     :",
    FULL_BEST_EPOCH
)

print(
    "Reason                  : "
    "Highest Dice in the supplied 30-epoch run"
)

print(
    f"Best Dice               : "
    f"{FULL_DICE:.4f}"
)

print(
    f"IoU at Dice-best epoch  : "
    f"{FULL_IOU:.4f}"
)

print(
    f"Accuracy at Dice-best epoch : "
    f"{FULL_ACCURACY:.4f}"
)

print(
    "\nNote: IoU reaches 0.8774 at Epoch 30 and "
    "Accuracy reaches 0.9891 at Epoch 20."
)

print(
    "For a consistent primary baseline, these independently "
    "higher values are not mixed with the Dice-best result."
)

print(
    "All three reported FULL baseline metrics above correspond "
    "to the same Dice-selected Epoch 29."
)


# ============================================================
# 9. CURRENT PRUNED DATASET RESULT
# ============================================================

print("\nPRUNED DATASET")
print("-" * 70)

print(
    f"Training images : "
    f"{PRUNED_TRAIN_IMAGES}"
)

print(
    f"Training time   : "
    f"{PRUNED_TRAINING_TIME:.2f} sec"
)

print(
    f"Best epoch      : "
    f"{PRUNED_BEST_EPOCH}"
)

print(
    f"Dice            : "
    f"{PRUNED_DICE:.4f}"
)

print(
    f"IoU             : "
    f"{PRUNED_IOU:.4f}"
)

print(
    f"Accuracy        : "
    f"{PRUNED_ACCURACY:.4f}"
)


# ============================================================
# 9A. PRUNED DATASET PERFORMANCE RETENTION
# ============================================================

print("\nPRUNED PERFORMANCE RETENTION")
print("-" * 70)

print(
    f"Dice performance retained     : "
    f"{dice_performance_retention:.2f}%"
)

print(
    f"IoU performance retained      : "
    f"{iou_performance_retention:.2f}%"
)

print(
    f"Accuracy performance retained : "
    f"{accuracy_performance_retention:.2f}%"
)


# ============================================================
# 10. COMPARISON
# ============================================================

print("\nCOMPARISON WITH FULL DATASET")
print("-" * 70)

print(
    f"Full training images     : "
    f"{FULL_TRAIN_IMAGES}"
)

print(
    f"Pruned training images   : "
    f"{PRUNED_TRAIN_IMAGES}"
)

print(
    f"Retention                : "
    f"{actual_retention * 100:.2f}%"
)

print(
    f"Data removed             : "
    f"{pruning_rate * 100:.2f}%"
)

print(
    f"Images removed           : "
    f"{images_removed}"
)


# ------------------------------------------------------------
# PERFORMANCE CHANGE
# ------------------------------------------------------------

print(
    f"\nFull Dice                : "
    f"{FULL_DICE:.4f}"
)

print(
    f"Pruned Dice              : "
    f"{PRUNED_DICE:.4f}"
)

print(
    f"Dice change              : "
    f"{dice_change:+.4f}"
)

print(
    f"Absolute Dice drop       : "
    f"{dice_drop:.4f}"
)


print(
    f"\nFull IoU                 : "
    f"{FULL_IOU:.4f}"
)

print(
    f"Pruned IoU               : "
    f"{PRUNED_IOU:.4f}"
)

print(
    f"IoU change               : "
    f"{iou_change:+.4f}"
)

print(
    f"Absolute IoU drop        : "
    f"{iou_drop:.4f}"
)


print(
    f"\nFull Accuracy            : "
    f"{FULL_ACCURACY:.4f}"
)

print(
    f"Pruned Accuracy          : "
    f"{PRUNED_ACCURACY:.4f}"
)

print(
    f"Accuracy change          : "
    f"{accuracy_change:+.4f}"
)


# ------------------------------------------------------------
# PERFORMANCE RETENTION
# ------------------------------------------------------------

print(
    f"\nDice performance retained     : "
    f"{dice_performance_retention:.2f}%"
)

print(
    f"IoU performance retained      : "
    f"{iou_performance_retention:.2f}%"
)

print(
    f"Accuracy performance retained : "
    f"{accuracy_performance_retention:.2f}%"
)


# ------------------------------------------------------------
# TRAINING EFFICIENCY
# ------------------------------------------------------------

print(
    f"\nFull training time       : "
    f"{FULL_TRAINING_TIME:.2f} sec"
)

print(
    f"Pruned training time     : "
    f"{PRUNED_TRAINING_TIME:.2f} sec"
)

print(
    f"Training time saved      : "
    f"{training_time_saved:.2f} sec"
)

print(
    f"Training speedup         : "
    f"{training_speedup:.2f}x"
)

print(
    f"Training time reduction  : "
    f"{training_time_reduction * 100:.2f}%"
)


# ============================================================
# 10A. RESULT INTERPRETATION
# ============================================================

print("\nRESULT INTERPRETATION")
print("-" * 70)

if dice_change > 0:
    print(
        "Dice interpretation      : "
        "Pruned dataset improved Dice compared with FULL."
    )
elif dice_change < 0:
    print(
        "Dice interpretation      : "
        "Pruned dataset reduced Dice compared with FULL."
    )
else:
    print(
        "Dice interpretation      : "
        "Pruned dataset produced the same Dice as FULL."
    )


if iou_change > 0:
    print(
        "IoU interpretation       : "
        "Pruned dataset improved IoU compared with FULL."
    )
elif iou_change < 0:
    print(
        "IoU interpretation       : "
        "Pruned dataset reduced IoU compared with FULL."
    )
else:
    print(
        "IoU interpretation       : "
        "Pruned dataset produced the same IoU as FULL."
    )


if accuracy_change > 0:
    print(
        "Accuracy interpretation  : "
        "Pruned dataset improved Accuracy compared with FULL."
    )
elif accuracy_change < 0:
    print(
        "Accuracy interpretation  : "
        "Pruned dataset reduced Accuracy compared with FULL."
    )
else:
    print(
        "Accuracy interpretation  : "
        "Pruned dataset produced the same Accuracy as FULL."
    )


# ============================================================
# 11. THESIS SUMMARY
# ============================================================

print("\nTHESIS SUMMARY")
print("-" * 70)

print(
    f"Model: "
    f"PVT-CASCADE"
)

print(
    f"Full dataset: "
    f"{FULL_TRAIN_IMAGES} training images"
)

print(
    f"Pruned dataset: "
    f"{PRUNED_TRAIN_IMAGES} training images"
)

print(
    f"Dataset reduction: "
    f"{pruning_rate * 100:.2f}%"
)

print(
    f"Dataset retention: "
    f"{actual_retention * 100:.2f}%"
)

print(
    f"\nFull Dice: "
    f"{FULL_DICE:.4f}"
)

print(
    f"Pruned Dice: "
    f"{PRUNED_DICE:.4f}"
)

print(
    f"Dice difference: "
    f"{dice_change:+.4f}"
)

print(
    f"Dice performance retained: "
    f"{dice_performance_retention:.2f}%"
)


print(
    f"\nFull IoU: "
    f"{FULL_IOU:.4f}"
)

print(
    f"Pruned IoU: "
    f"{PRUNED_IOU:.4f}"
)

print(
    f"IoU difference: "
    f"{iou_change:+.4f}"
)

print(
    f"IoU performance retained: "
    f"{iou_performance_retention:.2f}%"
)


print(
    f"\nFull Accuracy: "
    f"{FULL_ACCURACY:.4f}"
)

print(
    f"Pruned Accuracy: "
    f"{PRUNED_ACCURACY:.4f}"
)

print(
    f"Accuracy difference: "
    f"{accuracy_change:+.4f}"
)

print(
    f"Accuracy performance retained: "
    f"{accuracy_performance_retention:.2f}%"
)


print(
    f"\nFull training time: "
    f"{FULL_TRAINING_TIME:.2f} sec"
)

print(
    f"Pruned training time: "
    f"{PRUNED_TRAINING_TIME:.2f} sec"
)

print(
    f"Training speedup: "
    f"{training_speedup:.2f}x"
)

print(
    f"Training time reduction: "
    f"{training_time_reduction * 100:.2f}%"
)


# ============================================================
# 12. THESIS-ORIENTED DATA POINTS
# ============================================================
# These values are printed separately so that they are easy
# to copy into Excel, Google Sheets, thesis tables, or plots.


print("\nTHESIS DATA POINTS")
print("-" * 70)

print(
    f"Dataset                 : FULL"
)

print(
    f"Total images            : {FULL_DATASET_TOTAL_IMAGES}"
)

print(
    f"Training images         : {FULL_TRAIN_IMAGES}"
)

print(
    f"Test images             : {FULL_TEST_IMAGES}"
)

print(
    f"Retention               : 100.00%"
)

print(
    f"Pruning                 : 0.00%"
)

print(
    f"Best epoch              : {FULL_BEST_EPOCH}"
)

print(
    f"Dice                    : {FULL_DICE:.4f}"
)

print(
    f"IoU                     : {FULL_IOU:.4f}"
)

print(
    f"Accuracy                : {FULL_ACCURACY:.4f}"
)

print(
    f"Training time (sec)     : {FULL_TRAINING_TIME:.2f}"
)


print("\nPRUNED EXPERIMENT DATA POINT")

print(
    f"Dataset                 : PRUNED"
)

print(
    f"Training images         : {PRUNED_TRAIN_IMAGES}"
)

print(
    f"Retention               : "
    f"{actual_retention * 100:.2f}%"
)

print(
    f"Pruning                 : "
    f"{pruning_rate * 100:.2f}%"
)

print(
    f"Best epoch              : {PRUNED_BEST_EPOCH}"
)

print(
    f"Dice                    : {PRUNED_DICE:.4f}"
)

print(
    f"IoU                     : {PRUNED_IOU:.4f}"
)

print(
    f"Accuracy                : {PRUNED_ACCURACY:.4f}"
)

print(
    f"Training time (sec)     : "
    f"{PRUNED_TRAINING_TIME:.2f}"
)


# ============================================================
# 13. SAVED FILES
# ============================================================

print("\nSAVED FILES")
print("-" * 70)

print(
    "DINO features        :",
    config.FEATURE_FILE
)

print(
    "Pruned indices       :",
    config.PRUNED_INDEX_FILE
)

print(
    "Pruned model         :",
    pruned_result["checkpoint"]
)

# ============================================================
# 14. COMMUNITY STATUS
# ============================================================

print("\nCOMMUNITY STATUS")
print("-" * 70)

community_statistics = pruner.community_statistics

community_sizes = [
    item["size"]
    for item in community_statistics
]

print(
    "Number of communities:",
    len(community_statistics)
)

for item in community_statistics:

    print(
        f"Community {item['community_id']}: "
        f"{item['size']} samples, "
        f"{item['selected']} selected"
    )


# ============================================================
# 15. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)

print(
    "FULL DATASET BASELINE WAS USED AS A FIXED REFERENCE"
)

print(
    "ONLY THE PRUNED DATASET WAS TRAINED IN THIS EXPERIMENT"
)

print(
    f"FULL BASELINE DICE = {FULL_DICE:.4f}"
)

print(
    f"FULL BASELINE IoU  = {FULL_IOU:.4f}"
)

print(
    f"FULL BASELINE ACC  = {FULL_ACCURACY:.4f}"
)

print(
    f"FULL BASELINE EPOCH = {FULL_BEST_EPOCH}"
)

print("=" * 70)

print("\nDONE.")



FINAL THESIS RESULTS

CONFIGURATION
----------------------------------------------------------------------
Model                   : PVT-CASCADE
Primary metric          : Dice
Image size              : 224
Batch size              : 16
Learning rate           : 0.0001
Weight decay            : 0.0001
Full dataset epochs     : 30
Pruned dataset epochs   : 30
Random seed             : Not specified
DINO model              : rA9del/dinov3b16
Similarity threshold    : 1
Requested retention     : 12.00%

DATASET
----------------------------------------------------------------------
Total images            : 1000
Full training images    : 800
Pruned training images  : 489
Test images             : 200
Actual retention        : 61.12%
Images removed          : 311
Pruning rate            : 38.88%

FULL DATASET BASELINE
----------------------------------------------------------------------
Training images : 800
Training time   : 541.04 sec
Best epoch      : 26
Dice            : 0.9326
IoU    

# END